<a href="https://colab.research.google.com/github/supriyamishra0702-supmi/Deep-Learning-for-Comment-Toxicity-Detection-with-Streamlit/blob/main/Comment_Toxicity_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import nltk

from nltk.corpus import stopwords
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TextVectorization, Embedding, Bidirectional, LSTM, Dense


In [32]:
# Verifying that Colab is connected to the GPU
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [23]:
# Read both csv files
df_train = pd.read_csv('/content/train.csv', engine='python', on_bad_lines='skip')
df_test = pd.read_csv('/content/test.csv', engine='python', on_bad_lines='skip')




In [33]:

# Output spatial metadata metrics
print("Training Data Layout Dimensions:", df_train.shape)
print("Testing Data Layout Dimensions:", df_test.shape)




Training Data Layout Dimensions: (159571, 8)
Testing Data Layout Dimensions: (153164, 3)


In [34]:
# Review the first 3 rows of the training set
df_train.head(3)

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0


In [35]:
# Identify target toxicity labels
label_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']




In [37]:
print("--- Missing Values Report ---")
print(df_train['comment_text'].isnull().sum(), "missing entries found.")

print("\n--- Distribution Metrics Per Toxicity Category ---")
print(df_train[label_columns].sum())


--- Missing Values Report ---
0 missing entries found.

--- Distribution Metrics Per Toxicity Category ---
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64


In [38]:
# Download the standard lexicon directory of English stop words
nltk.download('stopwords')
stop_words_directory = set(stopwords.words('english'))



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [39]:
def learner_clean_text(raw_text):
    # 1. Normalize character casing
    text = str(raw_text).lower()

    # 2. Extract web addresses and hypertext protocols
    text = re.sub(r'http\S+', '', text)

    # 3. Filter structural characters (keep only pure lowercase alphabets and spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # 4. Break sentence down into list elements
    individual_words = text.split()

    # 5. Iteratively filter out non-essential stop words
    filtered_list = []
    for word in individual_words:
        if word not in stop_words_directory:
            filtered_list.append(word)

    # 6. Stitch words back together with regular spacing
    return " ".join(filtered_list)

print("Text preprocessing function successfully compiled!")


Text preprocessing function successfully compiled!


In [40]:
print("Preprocessing training entries... ")
df_train['cleaned_text'] = df_train['comment_text'].apply(learner_clean_text)

print("Preprocessing testing entries...")
df_test['cleaned_text'] = df_test['comment_text'].apply(learner_clean_text)

print("Data processing complete!")
df_train[['comment_text', 'cleaned_text']].head(3)



Preprocessing training entries... 
Preprocessing testing entries...
Data processing complete!


,comment_text,cleaned_text
0,Explanation\nWhy the edits made under my usern...,explanation edits made username hardcore metal...
1,D'aww! He matches this background colour I'm s...,daww matches background colour im seemingly st...
2,"Hey man, I'm really not trying to edit war. It...",hey man im really trying edit war guy constant...


In [41]:
# Separate structural labels from input data strings
X_train_text = df_train['cleaned_text'].values
y_train = df_train[label_columns].values
X_test_text = df_test['cleaned_text'].values



In [42]:
# Configure parameters for mapping words to numeric indices
max_dictionary_size = 50000
max_comment_length = 200



In [43]:
# Create the vectorization layer
vectorizer_layer = TextVectorization(
    max_tokens=max_dictionary_size,
    output_sequence_length=max_comment_length,
    output_mode='int'
)





In [44]:
# Build the word index mapping based exclusively on training text
vectorizer_layer.adapt(X_train_text)

# Convert clean text sentences into numeric sequences
X_train = vectorizer_layer(X_train_text).numpy()
X_test = vectorizer_layer(X_test_text).numpy()



In [45]:

print("Vectorization execution completed!")
print("X_train Matrix Dimensions:", X_train.shape)
print("X_test Matrix Dimensions:", X_test.shape)


Vectorization execution completed!
X_train Matrix Dimensions: (159571, 200)
X_test Matrix Dimensions: (153164, 200)


In [47]:
# Construct the model using a step-by-step layer stack
model = Sequential([
    # Layer 1: Learn rich vector space meanings for words
    Embedding(input_dim=max_dictionary_size, output_dim=32, input_length=max_comment_length),

    # Layer 2: Read word paths forward and backward to catch contextual nuances
    Bidirectional(LSTM(32, activation='tanh')),

    # Layer 3: Uncover hidden structural relationships
    Dense(64, activation='relu'),

    # Layer 4: Generate separate classification probabilities for the 6 labels
    Dense(6, activation='sigmoid')
])

# Define performance monitoring rules
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Neural Architecture successfully constructed!")


Neural Architecture successfully constructed!


In [48]:
print("Executing deep learning loop routines on GPU...")

# Run training loops across the dataset
history = model.fit(
    X_train,
    y_train,
    epochs=3,          # Pass through the full training data 3 times
    batch_size=64,     # Evaluate 64 samples before adjusting internal model weights
    shuffle=True       # Scramble rows after each epoch to prevent memorisation
)

print("Training phase successfully completed!")


Executing deep learning loop routines on GPU...
Epoch 1/3
2494/2494 ━━━━━━━━━━━━━━━━━━━━ 57s 20ms/step - accuracy: 0.9811 - loss: 0.0705
Epoch 2/3
2494/2494 ━━━━━━━━━━━━━━━━━━━━ 75s 19ms/step - accuracy: 0.9942 - loss: 0.0449
Epoch 3/3
2494/2494 ━━━━━━━━━━━━━━━━━━━━ 48s 19ms/step - accuracy: 0.9939 - loss: 0.0403
Training phase successfully completed!


In [49]:
# Create the local folder structure
os.makedirs('models', exist_ok=True)

# 1. Save the trained neural weights
model.save('models/toxicity_lstm_model.keras')

# 2. Save the vocabulary configuration file for your Streamlit application
vectorizer_data = {
    'config': vectorizer_layer.get_config(),
    'weights': vectorizer_layer.get_weights()
}

with open('models/vectorizer_config.pkl', 'wb') as file:
    pickle.dump(vectorizer_data, file)

print("Model and vectorizer saved successfully inside the 'models' directory!")


Model and vectorizer saved successfully inside the 'models' directory!
